# Brain Tumor Classification using Deep Learning (EfficientNetV2)

**Author:** Khalid Azimi

**Objective:** Develop a robust AI model to classify MRI scans into 4 categories: Glioma, Meningioma, Pituitary Tumor, and No Tumor, achieving >94% accuracy using Transfer Learning and Test-Time Augmentation (TTA).

---

## 📋 Table of Contents

*   Environment & Data Acquisition
*   Exploratory Data Analysis (EDA)
*   Data Pipeline & Augmentation Strategy
*   Model Architecture (EfficientNetV2B3)
*   Two-Phase Training Strategy
*   Model Evaluation & Confusion Matrix
*   Real-Time Inference Demo

In [ ]:
# ==========================================
# 1. ENVIRONMENT & DATA ACQUISITION
# ==========================================
import os
import tensorflow as tf
from google.colab import files

print("⚙️ Installing dependencies...")
!pip install -q kaggle

print("📤 Upload your kaggle.json file (from Kaggle > Settings > API)...")
uploaded = files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

print("⬇️ Downloading Brain Tumor MRI Dataset...")
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d /content/data && rm brain-tumor-mri-dataset.zip

# Verify dataset
data_dir = '/content/data'
classes = os.listdir(f"{data_dir}/Training")
print("\n✅ Dataset loaded successfully!")
print(f"📂 Classes: {classes}")

In [ ]:
# ==========================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import random

# Class distribution
train_counts = [len(os.listdir(f"{data_dir}/Training/{c}")) for c in classes]
test_counts = [len(os.listdir(f"{data_dir}/Testing/{c}")) for c in classes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

axes[0].bar(classes, train_counts, color=colors, edgecolor='black')
axes[0].set_title('Training Set Distribution', fontweight='bold')
for i, v in enumerate(train_counts): axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].bar(classes, test_counts, color=colors, edgecolor='black')
axes[1].set_title('Testing Set Distribution', fontweight='bold')
for i, v in enumerate(test_counts): axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Sample Images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(classes):
    img_path = f"{data_dir}/Training/{cls}/{random.choice(os.listdir(f'{data_dir}/Training/{cls}'))}"
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(cls.upper(), fontweight='bold', color=colors[i])
    axes[i].axis('off')
plt.suptitle('Sample MRI Scans per Class', fontsize=14, fontweight='bold', y=1.05)
plt.show()

In [ ]:
# ==========================================
# 3 & 4. DATA PIPELINE & MODEL ARCHITECTURE
# ==========================================
from tensorflow.keras.applications import EfficientNetV2B3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Preprocessing optimized for EfficientNetV2
preprocess_input = tf.keras.applications.efficientnet_v2.preprocess_input
IMG_SIZE = 224
BATCH_SIZE = 32

# High-performance tf.data pipeline
train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{data_dir}/Training', image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    shuffle=True, seed=42, label_mode='categorical'
).map(lambda x, y: (preprocess_input(x), y)).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{data_dir}/Testing', image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    shuffle=False, label_mode='categorical'
).map(lambda x, y: (preprocess_input(x), y)).prefetch(tf.data.AUTOTUNE)

# Model Definition
base_model = EfficientNetV2B3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Phase 1 freeze

data_aug = tf.keras.Sequential([RandomFlip("horizontal_and_vertical"), RandomRotation(0.2), RandomZoom(0.15)])

inputs = Input(shape=(224, 224, 3))
x = data_aug(inputs)
x = base_model(x)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(4, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print("✅ Model built successfully.")

In [ ]:
# ==========================================
# 5. TWO-PHASE TRAINING STRATEGY
# ==========================================
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
import time

callbacks = [
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy', mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
]

# Phase 1: Train classification head
print("⚡ Phase 1: Training Head (10 Epochs)...")
t0 = time.time()
history1 = model.fit(train_ds, epochs=10, validation_data=val_ds, verbose=1)
print(f"⏱️ Phase 1 done in {time.time()-t0:.0f} sec\n")

# Phase 2: Fine-tune backbone
print("🔥 Phase 2: Fine-Tuning Backbone...")
base_model.trainable = True
model.compile(optimizer=Adam(5e-5), loss='categorical_crossentropy', metrics=['accuracy'])

t1 = time.time()
history2 = model.fit(train_ds, epochs=25, validation_data=val_ds, callbacks=callbacks, verbose=1)
print(f"⏱️ Phase 2 done in {time.time()-t1:.0f} sec")

# Plot Training Curves
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(val_acc, color='red', lw=2); ax[0].set_title('Validation Accuracy', fontweight='bold'); ax[0].grid(True, alpha=0.3)
ax[1].plot(val_loss, color='blue', lw=2); ax[1].set_title('Validation Loss', fontweight='bold'); ax[1].grid(True, alpha=0.3)
plt.show()
print(f"🏆 Best Validation Accuracy: {max(val_acc)*100:.2f}%")

In [ ]:
# ==========================================
# 6. ADVANCED EVALUATION & TTA
# ==========================================
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Load absolute best model
model = tf.keras.models.load_model('best_model.keras')

# Standard Evaluation
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels, axis=1)); y_pred.extend(np.argmax(preds, axis=1))

# Test-Time Augmentation (TTA) for final edge-case smoothing
y_tta = []
for images, labels in val_ds:
    p1 = model.predict(images, verbose=0)
    p2 = model.predict(tf.image.flip_left_right(images), verbose=0)
    p3 = model.predict(tf.image.flip_up_down(images), verbose=0)
    y_tta.extend(np.argmax(np.mean([p1, p2, p3], axis=0), axis=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# Standard CM
cm1 = confusion_matrix(y_true, y_pred)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=class_names, yticklabels=class_names)
axes[0].set_title(f'Standard Evaluation ({np.mean(np.array(y_true)==np.array(y_pred))*100:.2f}%)', fontweight='bold')

# TTA CM
cm2 = confusion_matrix(y_true, y_tta)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=axes[1], xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'TTA Evaluation ({np.mean(np.array(y_true)==np.array(y_tta))*100:.2f}%)', fontweight='bold')

for ax in axes: ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("\n📊 Final TTA Classification Report:")
print(classification_report(y_true, y_tta, target_names=class_names, digits=4))

In [ ]:
# ==========================================
# 7. REAL-TIME INFERENCE DEMO
# ==========================================
from tensorflow.keras.preprocessing import image

def diagnose_mri(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
    prediction = model.predict(img_array, verbose=0)[0]

    plt.figure(figsize=(6,5))
    bars = plt.barh(class_names, prediction*100, color=['#FF6B6B', '#FFA500', '#2ecc71', '#3498db'])
    plt.xlim(0, 105); plt.title("AI Diagnosis Confidence", fontweight='bold')
    for bar, p in zip(bars, prediction): plt.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2, f'{p*100:.1f}%', va='center', fontweight='bold')
    plt.show()
    print(f"🩺 Final Diagnosis: {class_names[np.argmax(prediction)].upper()} ({max(prediction)*100:.2f}% confidence)")

print("📤 Upload an MRI image to test the AI:")
uploaded = files.upload()
for fn in uploaded.keys(): diagnose_mri(fn)